In [ ]:
! pip3 install -r requirements.txt

In [1]:
!nvidia-smi

Wed Nov 15 13:37:23 2023       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 525.105.17   Driver Version: 525.105.17   CUDA Version: 12.0     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  Tesla T4            Off  | 00000000:00:04.0 Off |                    0 |
| N/A   57C    P0    27W /  70W |      2MiB / 15360MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
|   1  Tesla T4            Off  | 00000000:00:05.0 Off |                    0 |
| N/A   

In [2]:
import os
from utils.utils import get_offers
from utils.secrets_utils import get_secret
from utils.constants import GCP_PROJECT_ID,ENV_SHORT_NAME,BIGQUERY_ANALYTICS_DATASET
import vertexai
from google.cloud import aiplatform
import json
from vertexai.language_models import ChatModel, InputOutputTextPair


In [3]:
offer_list = get_offers(number_offers=500,analytics_dataset="analytics_stg")

/home/airflow/.local/lib/python3.10/site-packages/google/auth/_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Downloading: 100%|██████████|


In [4]:
# Cloud project id.
PROJECT_ID = GCP_PROJECT_ID,ENV_SHORT_NAME # @param {type:"string"}

# The region you want to launch jobs in.
# Select region based on the accelerators and regions supported by Vertex AI Prediction
# https://cloud.google.com/vertex-ai/docs/predictions/configure-compute.
REGION = "us-central1"  # @param {type:"string"}

# The Cloud Storage bucket for storing experiments output.
# Start with gs:// prefix, e.g. gs://foo_bucket.
BUCKET_URI = "gs://data-bucket-stg"  # @param {type:"string"}

! gcloud config set project $GCP_PROJECT_ID


STAGING_BUCKET = os.path.join(BUCKET_URI, "temporal")

# The service account looks like:
# '@.iam.gserviceaccount.com'
# Please go to https://cloud.google.com/iam/docs/service-accounts-create#iam-service-accounts-create-console
# and create service account with `Vertex AI User` and `Storage Object Admin` roles.
# The service account for deploying fine tuned model.
SERVICE_ACCOUNT = "cloud-run-dev@passculture-data-ehp.iam.gserviceaccount.com"  # @param {type:"string"}


To update your Application Default Credentials quota project, use the `gcloud auth application-default set-quota-project` command.
Updated property [core/project].


In [5]:
aiplatform.init(project=PROJECT_ID, location=REGION, staging_bucket=STAGING_BUCKET)

In [6]:
# %%time
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda"  # the device to load the model onto
model_name = "mistralai/Mistral-7B-v0.1" # "mistralai/Mistral-7B-Instruct-v0.1" -> specialized in english and code
model = AutoModelForCausalLM.from_pretrained(
    model_name, device_map="auto", return_dict=True, torch_dtype=torch.float16
)
tokenizer = AutoTokenizer.from_pretrained(model_name)





Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [8]:
EXTRACT_ENTITIES = """En tant qu'expert rédacteur éditorial spécialisé dans la création de listes recommandation d'offres d'activitées et de pratiques culturelles, 
votre tâche consiste à analyser avec soin les offres culturelles afin d'en extaire les entités culturelles. Ces entitées peuvent être des:
- une liste des artistes et leur metier sous format {"nom d'artiste": "métier"} présents dans l'offre, voici des exemples de metier:  (écrivain, metteur en scène,acteur , réalisateur, performer, groupe de musique, peintre, photographe, etc...), 
- une liste des disciplines artistiques majeures et champs culturels présents dans l'offre, par exemple (philosophie, peinture, littérature, cinéma, danse, sculpture, dessin, musique, mode,etc...)
- une liste des lieux ou évènements présents dans l'offre, par exemple (musées, monuments, festivals, rétrospectives, etc...)
- une liste des oeuvres ou des personnages principaux présents dans l'offre,
- une liste des spécificités des pratiques artistiques et culturelles ou les thème abordés dans l'offre, par exemple (courant artistique, genre musical, genre cinématographique, etc...).
Afin de répondre précisément à mes besoins, veuillez répondre à ces questions sous format JSON object en associant a chacune de ces typologie d'entités une liste d'un ou plusieurs résultats s'ils existent sinon associer la valeur []. N'inventez pas !
"""

In [9]:
CONTEXT = 
Q_ENTITIES = """
- quels sont les artistes et leur metier sous format {"nom d'artiste": "métier"} présents dans l'offre? Voici des exemples de metiers:  (écrivain, metteur en scène,acteur , réalisateur, performer, groupe de musique, peintre, photographe, etc...), 
- quels sont les disciplines artistiques majeures et champs culturels présents dans l'offre? Voici des exemples (philosophie, peinture, littérature, cinéma, danse, sculpture, dessin, musique, mode,etc...)
- quels sont les lieux ou évènements présents dans l'offre? Voici des exemples (musée, monument, festival, rétrospective, etc...)
- quels sont les oeuvres ou des personnages principaux présents dans l'offre?
- quels sont les spécificités des pratiques artistiques et culturelles ou les thème abordés dans l'offre? Voici des exemples (courant artistique, genre musical, genre cinématographique, etc...).
Afin de répondre précisément à mes besoins, veuillez répondre à ces questions sous format JSON object en associant a chacune de ces typologie d'entités une liste d'un ou plusieurs résultats s'ils existent sinon associer la valeur [].
"""
questions = ["""Quels sont les artistes et leur metier sous format {"nom d'artiste": "métier"} présents dans le document? Voici des exemples de metiers:  (écrivain, metteur en scène,acteur , réalisateur, performer, groupe de musique, peintre, photographe, etc...)""",
             """Quels sont les disciplines artistiques majeures et champs culturels présents dans le document? Voici des exemples (philosophie, peinture, littérature, cinéma, danse, sculpture, dessin, musique, mode,etc...)""",
             """Quels sont les lieux ou évènements présents dans le document? Voici des exemples (musée, monument, festival, rétrospective, etc...)""",
             """Quels sont les oeuvres ou des personnages principaux présents dans le document?""",
             """Quels sont les spécificités des pratiques artistiques et culturelles ou les thème abordés dans le document? Voici des exemples (courant artistique, genre musical, genre cinématographique, etc...)."""
             ]

In [10]:
prompt = f"""user: {EXTRACT_ENTITIES} Voici l'offre: {offer_list[2]}"""
model(
    prompt)

AttributeError: 'str' object has no attribute 'shape'

tasks to look into : "text-generation", "conversational", "document-question-answering","feature-extraction","question-answering", "summarization"

In [20]:
task = "text-generation"
pipeline = transformers.pipeline(model=model, tokenizer=tokenizer)


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


RuntimeError: Inferring the task automatically requires to check the hub with a model_id defined as a `str`.MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32000, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): MistralRotaryEmbedding()
        )
        (mlp): MistralMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm

### https://huggingface.co/blog/accelerate-large-models

In [19]:



sequences = pipeline(
    prompt,
    max_length=512,
    do_sample=True,
    top_k=10,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
)

for seq in sequences:
    print(f"Result: {seq['generated_text']}")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
/home/airflow/.local/lib/python3.10/site-packages/transformers/generation/utils.py:1268: UserWarning: Input length of input_ids is 730, but `max_length` is set to 512. This can lead to unexpected behavior. You should consider increasing `max_new_tokens`.
  warnings.warn(


Result: user: En tant qu'expert rédacteur éditorial spécialisé dans la création de listes recommandation d'offres d'activitées et de pratiques culturelles, 
votre tâche consiste à analyser avec soin les offres culturelles afin d'en extaire les entités culturelles. Ces entitées peuvent être des:
- une liste des artistes et leur metier sous format {"nom d'artiste": "métier"} présents dans l'offre, voici des exemples de metier:  (écrivain, metteur en scène,acteur , réalisateur, performer, groupe de musique, peintre, photographe, etc...), 
- une liste des disciplines artistiques majeures et champs culturels présents dans l'offre, par exemple (philosophie, peinture, littérature, cinéma, danse, sculpture, dessin, musique, mode,etc...)
- une liste des lieux ou évènements présents dans l'offre, par exemple (musées, monuments, festivals, rétrospectives, etc...)
- une liste des oeuvres ou des personnages principaux présents dans l'offre,
- une liste des spécificités des pratiques artistiques et 

In [30]:
PROMPT= f""" ### Instruction: En tant qu'expert rédacteur éditorial spécialisé dans la création de listes recommandation d'offres d'activitées et de pratiques culturelles, 
votre tâche consiste à analyser avec soin l'offre culturelle suivante afin d'en extaire les entités culturelles.
Afin de répondre précisément à mes besoins, veuillez répondre à ces questions sous format JSON object en associant a chacune de ces typologie d'entités une liste d'un ou plusieurs résultats s'ils existent sinon associer la valeur [].
Voici l'offre: {offer_list[2]}.

### Question:
{Q_ENTITIES}

### Answer:
"""

encodeds = tokenizer(PROMPT, return_tensors="pt", add_special_tokens=True)
model_inputs = encodeds.to(device)

generated_ids = model.generate(**model_inputs, max_new_tokens=512, do_sample=True, pad_token_id=tokenizer.eos_token_id)
decoded = tokenizer.batch_decode(generated_ids)
print(decoded[0])

OutOfMemoryError: CUDA out of memory. Tried to allocate 82.00 MiB. GPU 1 has a total capacty of 14.58 GiB of which 43.38 MiB is free. Including non-PyTorch memory, this process has 14.53 GiB memory in use. Of the allocated memory 13.58 GiB is allocated by PyTorch, and 263.10 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [10]:
pipeline = transformers.pipeline(model=model, tokenizer=tokenizer)

task = "feature-extraction" #, "summarization"
model_name = "mistralai/Mistral-7B-v0.1"
model = AutoModelForCausalLM.from_pretrained(
    model_name, device_map="auto", return_dict=True, torch_dtype=torch.float16
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

feature_extractor = pipeline(task, model=model, tokenizer=tokenizer)


preds = feature_extractor(
    question=f"{EXTRACT_ENTITIES}",
    context=f"{offer_list[2]}",
)

print(
    f"score: {round(preds['score'], 4)}, start: {preds['start']}, end: {preds['end']}, answer: {preds['answer']}")


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


ValueError: The following `model_kwargs` are not used by the model: ['model', 'tokenizer'] (note: typos in the generate arguments will also show up in this list)

In [12]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

model_name_or_path = "mistralai/Mistral-7B-Instruct-v0.1"
# To use a different branch, change revision
model = AutoModelForCausalLM.from_pretrained(model_name_or_path,
                                             device_map="auto"
                                            )

tokenizer = AutoTokenizer.from_pretrained(model_name_or_path, use_fast=True)

prompt = EXTRACT_ENTITIES + f" Voici l'offre: {offer_list[2]}"
prompt_template=f'''<s>[INST] {prompt} [/INST]
'''

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    do_sample=True,
    temperature=0.7,
    top_p=0.95,
    top_k=40,
    repetition_penalty=1.1
)

print(pipe(prompt_template)[0]['generated_text'])
tok = tokenizer("prompt_template")
tokens = len(tok['input_ids'])
print(f"Number of tokens: {tokens}")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


OutOfMemoryError: CUDA out of memory. Tried to allocate 12.00 MiB. GPU 1 has a total capacty of 14.58 GiB of which 9.31 MiB is free. Including non-PyTorch memory, this process has 14.56 GiB memory in use. Of the allocated memory 13.76 GiB is allocated by PyTorch, and 127.40 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [13]:
def prompt_offer(i):
    prompt = EXTRACT_ENTITIES + f" Voici l'offre: {offer_list[i]}"
    prompt_template=f'''<s>[INST] {prompt} [/INST]f'''
    return prompt_template

In [14]:
print(pipe(prompt_offer(5))[0]['generated_text'])

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


OutOfMemoryError: CUDA out of memory. Tried to allocate 12.00 MiB. GPU 1 has a total capacty of 14.58 GiB of which 9.31 MiB is free. Including non-PyTorch memory, this process has 14.56 GiB memory in use. Of the allocated memory 13.76 GiB is allocated by PyTorch, and 127.40 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [17]:
!nvidia-smi


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Wed Nov 15 13:49:04 2023       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 525.105.17   Driver Version: 525.105.17   CUDA Version: 12.0     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  Tesla T4            Off  | 00000000:00:04.0 Off |                    0 |
| N/A   68C    P0    29W /  70W |  14567MiB / 15360MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
|   1  Tesla T4            Off  | 00000000:00:05.0 Off |                    0 |
| N/A   